# TV-00b C# — Variantes d’attention : MHA, MQA, GQA, SWA

Le notebook précédent (`TV-00a-RoPE-from-scratch.ipynb`) ouvrait la boîte du **positionnement** ; celui-ci ouvre celle de l’**attention** telle qu’elle est réellement câblée dans LLaMA-2/3, Mistral ou Qwen.

Un modèle comme Mistral-7B n’exécute pas `attention(Q, K, V)` avec 32 têtes identiques. Il utilise trois variantes que le Transformer canonique ne montre pas, chacune répondant à une contrainte de coût :

1. **MQA** (*Multi-Query Attention*) : une seule paire de projections K/V pour toutes les têtes Q. Le cache KV de décodage s’effondre d’un facteur égal au nombre de têtes.
2. **GQA** (*Grouped-Query Attention*) : les têtes Q sont réparties en groupes qui **partagent** une tête K/V. C’est le réglage de LLaMA-2 70B (64 têtes Q, 8 têtes KV) et de Mistral, un compromis entre la qualité de MHA et la mémoire de MQA.
3. **SWA** (*Sliding Window Attention*) : chaque requête ne lit que les `W` dernières clés. Le coût passe de quadratique à linéaire en la longueur, au prix de l’information hors fenêtre.

L’objectif est de **coder** ces variantes, puis de **mesurer** leur coût et leurs apports sur un banc dont chaque paramètre est contrôlé.

| Question | Instrument |
|---|---|
| Les variantes sont-elles correctes ? | Échelle de vérifications exactes : boucle par tête, invariants de groupement, identité de masque |
| Combien de mémoire le cache KV économise-t-il ? | Formule analytique confrontée à une allocation réelle |
| Le partage des têtes KV accélère-t-il l’entraînement ? | Chronométrage du passage avant et arrière |
| La fenêtre glissante accélère-t-elle vraiment ? | Bande explicite `O(T*W)` contre masque `O(T^2)`, avec balayage de `T` |
| Que coûte la fenêtre en qualité ? | Tâche de rappel à longue portée, où la distance dépasse la fenêtre |

Tout tourne sur CPU avec TorchSharp après restauration du package NuGet : aucun modèle externe ni `transformers`, `xformers` ou `flash-attn`.

[Version Python](TV-00b-Attention-Variants-from-scratch.ipynb) | [RoPE C#](TV-00a-RoPE-from-scratch-CSharp.ipynb)

## Sommaire

1. L'attention causale multi-tetes, et sa brique de reference
2. Le cache KV : le vrai cout a l'inference
3. GQA et MQA : partager les tetes KV
4. SWA : la bande locale
5. Vitesse : ce que le partage accelere, et ce qu'il n'accelere pas
6. Qualite : une tâche ou la fenêtre se paie cash
7. Conclusion et suite

**Exercices** (cellules stubbees a completer) : le groupement des tetes Q, le masque en bande, et le
balayage de fenêtre. Indices et étapes donnes dans le markdown qui precede chacun.

**Conventions.** Documentation en francais, code et identifiants en francais ou anglais. Determinisme
verifie : `torch.set_num_threads(1)` réduit les variations de réduction ; les latences restent sensibles à la charge machine. Aucune erreur volontaire dans les cellules d'exercice :
un stub renvoie `null` et le notebook s'execute de bout en bout.

In [1]:
#r "nuget:TorchSharp-cpu, 0.107.0"
using System;

using System.Linq;

using System.Collections.Generic;

using System.Diagnostics;

using TorchSharp;

using static TorchSharp.torch;

using static TorchSharp.torch.nn;

using Tensor=TorchSharp.torch.Tensor;

torch.set_num_threads(1);
 torch.manual_seed(0);

Console.WriteLine($"TorchSharp {typeof(torch).Assembly.GetName().Version}, libtorch {torch.__version__}, CPU, threads {torch.get_num_threads()}");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages TorchSharp-cpu, 0.107.0

Loading extensions from `/workspace/coursia-toolchains/nuget/skiasharp/2.88.6/interactive-extensions/dotnet/SkiaSharp.DotNet.Interactive.dll`

TorchSharp 0.107.0.0, libtorch 2.10.0.0, CPU, threads 1


## 1. L'attention causale multi-tetes

Rappel de la forme canonique, tete `h` comprise :

```
scores[h] = Q[h] K[h]^T / sqrt(d_head)
attn[h]   = softmax(mask_causal(scores[h]))
sortie[h] = attn[h] V[h]
```

avec `Q = x W_q`, `K = x W_k`, `V = x W_v`, une concatenation des tetes en sortie, puis `W_o`.
Le masque causal est triangulaire : la requête `i` ne lit que les clés `j <= i`. C'est ce masque qui
rend le modèle autoregressif, et c'est lui aussi qui rend le **cache KV** possible : a la generation,
la cle et la valeur de chaque position déjà produite ne changent plus, on les garde.

On code d'abord une brique de reference **non vectorisee** — une boucle explicite sur les tetes. Elle
est lente et c'est voulu : elle sert de verite contre laquelle verifier la forme vectorisee que tout
le reste du notebook utilise. Une implementation d'attention qui n'est jamais confrontee a une
seconde implementation independante est une implementation dont on ne sait rien.

In [2]:
public class Variant:Module<Tensor,Tensor> {
 readonly long h,g,dh;
 readonly int? window;
 readonly bool banded;

 readonly Module<Tensor,Tensor> q,k,v,o;

 public Variant(int d,int heads,int groups,int? w=null,bool band=false):base("Variant") {h=heads;
g=groups;
dh=d/heads;
window=w;
banded=band;
q=torch.nn.Linear(d,heads*dh,hasBias:false);
k=torch.nn.Linear(d,groups*dh,hasBias:false);
v=torch.nn.Linear(d,groups*dh,hasBias:false);
o=torch.nn.Linear(d,d,hasBias:false);
RegisterComponents();
}
 public static Tensor Mask(Tensor q,Tensor k,Tensor v,int? w){var t=q.shape[2];
var i=arange(t).reshape(t,1);
var j=arange(t).reshape(1,t);
var mask=j.le(i);
if(w.HasValue)mask=mask.logical_and((i-j).lt(w.Value));
var scores=matmul(q,k.transpose(-2,-1))/Math.Sqrt(q.shape[3]);
return matmul(scores.masked_fill(mask.logical_not(),double.NegativeInfinity).softmax(-1),v);
}
 public static Tensor Band(Tensor q,Tensor k,Tensor v,int window){var t=q.shape[2];
var w=Math.Min(window,(int)t);
 var pad=zeros(new long[]{q.shape[0],q.shape[1],w-1,q.shape[3]});
var kw=cat(new[]{pad,k},2).unfold(2,w,1);
var vw=cat(new[]{pad,v},2).unfold(2,w,1);
var scores=einsum("bhtd,bhtdw->bhtw",q,kw)/Math.Sqrt(q.shape[3]);
var keep=arange(w).reshape(1,1,1,w).ge((w-1-arange(t)).reshape(1,1,t,1));
return einsum("bhtw,bhtdw->bhtd",scores.masked_fill(keep.logical_not(),double.NegativeInfinity).softmax(-1),vw);
}
 public long KvParams() => k.parameters().Sum(p=>p.numel()) + v.parameters().Sum(p=>p.numel());

 public override Tensor forward(Tensor x){var b=x.shape[0];
var t=x.shape[1];
var Q=q.forward(x).reshape(b,t,h,dh).transpose(1,2);
var K=k.forward(x).reshape(b,t,g,dh).transpose(1,2);
var V=v.forward(x).reshape(b,t,g,dh).transpose(1,2);
if(g!=h){K=K.repeat_interleave(h/g,1);
V=V.repeat_interleave(h/g,1);
}var z=banded&&window.HasValue?Band(Q,K,V,window.Value):Mask(Q,K,V,window);
return o.forward(z.transpose(1,2).reshape(b,t,h*dh));
}
}

Tensor Naive(Tensor q,Tensor k,Tensor v){var heads=new List<Tensor>();
for(int h=0;h<q.shape[1];h++){var Q=q.select(1,h);
var K=k.select(1,h);
var V=v.select(1,h);
var t=q.shape[2];
var mask=torch.triu(torch.ones(new long[]{t,t},dtype:ScalarType.Bool),1);
var s=torch.matmul(Q,K.transpose(-2,-1))/Math.Sqrt(q.shape[3]);
heads.Add(torch.matmul(s.masked_fill(mask,double.NegativeInfinity).softmax(-1),V));
}return torch.stack(heads.ToArray(),1);
}
Console.WriteLine("Attention causale par tête, masque vectorisé et bande définis.");


Attention causale par tête, masque vectorisé et bande définis.


### 1.1 Echelle de verifications

Avant de mesurer quoi que ce soit, on etablit que les briques sont ce qu'elles pretendent etre. Cinq
verifications, chacune confrontant le code a une **propriete enoncee independamment** :

- **(a)** la forme vectorisee (reshape + broadcasting) reproduit la boucle tete par tete ;
- **(b)** une fenêtre au moins aussi grande que la sequence ne mord pas : le masque fenêtre est alors
  le masque causal pur, exactement ;
- **(c)** le nombre de paires (requête, cle) lues sous fenêtre `W` vaut exactement
  `T*W - W(W-1)/2` des que `T >= W` — la quantite qui fait passer le cout de `O(T^2)` a `O(T*W)` ;
- **(d)** l'etagement des tetes KV (section 3) place la tete Q `h` sur la tete KV `h // (H/G)` ;
- **(e)** la bande explicite de la section 4 redonne le résultat du masque, au bruit flottant pres.

Aucune de ces verifications n'utilise le résultat qu'elle valide. C'est le point : `(a)` et `(e)`
comparent deux chemins de calcul independants, `(b)` et `(c)` comparent le code a une formule
fermee, `(d)` a un invariant de groupement.

In [3]:
var inputs=torch.randn(new long[]{2,16,64});
var weightGen=new torch.Generator(3,torch.CPU);

var wq=torch.randn(new long[]{64,64},generator:weightGen)/Math.Sqrt(64);
var wk=torch.randn(new long[]{64,64},generator:weightGen)/Math.Sqrt(64);
var wv=torch.randn(new long[]{64,64},generator:weightGen)/Math.Sqrt(64);
var wo=torch.randn(new long[]{64,64},generator:weightGen)/Math.Sqrt(64);

var q=torch.matmul(inputs,wq).reshape(2,16,4,16).transpose(1,2);
var k=torch.matmul(inputs,wk).reshape(2,16,4,16).transpose(1,2);
var v=torch.matmul(inputs,wv).reshape(2,16,4,16).transpose(1,2);

var refH=torch.matmul(Naive(q,k,v).transpose(1,2).reshape(2,16,64),wo);
var vec=torch.matmul(Variant.Mask(q,k,v,null).transpose(1,2).reshape(2,16,64),wo);
var diff=(refH-vec).abs().max().item<float>();
Console.WriteLine($"(a) boucle / vectorisé : {diff:E3}");
 if(diff>1e-5)throw new Exception("attention par tête incorrecte");

foreach(var (T,W) in new[]{(64,64),(64,200)}){var i=torch.arange(T).reshape(T,1);
var j=torch.arange(T).reshape(1,T);
var a=j.le(i);
var b=a.logical_and((i-j).lt(W));
Console.WriteLine($"(b) T={T}, W={W}, différence masque : {(a.to_type(ScalarType.Int32)-b.to_type(ScalarType.Int32)).abs().sum().item<long>()}");
}
foreach(var (T,W) in new[]{(64,16),(64,32),(100,7)}){var i=torch.arange(T).reshape(T,1);
var j=torch.arange(T).reshape(1,T);
var n=j.le(i).logical_and((i-j).lt(W)).sum().item<long>();
var formula=T*W-W*(W-1)/2;
Console.WriteLine($"(c) T={T}, W={W} : {n}, formule {formula}");
if(n!=formula)throw new Exception("paires");
}

(a) boucle / vectorisé : 0.000E+000


(b) T=64, W=64, différence masque : 0


(b) T=64, W=200, différence masque : 0


(c) T=64, W=16 : 904, formule 904


(c) T=64, W=32 : 1552, formule 1552


(c) T=100, W=7 : 679, formule 679


### 1.2 Lecture

Les contrôles comparent la boucle par tête et le calcul vectorisé, puis vérifient les masques et le comptage des paires. Les écarts doivent rester sous la tolérance.

## 2. Le cache KV : le vrai cout a l'inference

A l'entraînement, les quatre variantes ont le **même nombre d'opérations** : on calcule l'attention
sur toute la sequence, pour toutes les requêtes, avec ou sans partage des tetes K/V. Le partage ne
reduit pas les FLOPs d'attention ; il reduit la taille des **projections** K/V et, surtout, la taille
du cache a la generation.

A la generation autoregressive, a chaque nouveau jeton on ajoute une cle et une valeur par tete K/V
et par couche, et on les relit toutes. La taille du cache vaut :

```
octets = 2 (K et V) x B x n_tetes_KV x T x d_head x taille_element x n_couches
```

Le facteur `2` est le couple K/V, le `2` suivant (implicite) vient de la precision : un cache en
`float16` coute deux octets par élément. Ce sont `n_tetes_KV` et `T` qui decident, et `T` croit
lineairement avec la conversation. C'est cette formule qui explique pourquoi LLaMA-2 70B a ete
publie avec 8 tetes KV pour 64 tetes Q.

On calcule la formule sur des configurations **reelles** plutot que sur des ordres de grandeur, puis
on verifie la formule contre une allocation effective.

In [4]:
long Cache(long layers,long groups,long dh,long T,long batch=1,long octets=2)=>2*batch*layers*groups*T*dh*octets;

foreach(var (name,d,h,g,l) in new[]{("LLaMA-2 7B",4096,32,32,32),("LLaMA-2 70B",8192,64,8,80),("Mistral 7B",4096,32,8,32),("notre jouet",64,4,4,2)}){var mha=Cache(l,h,d/h,32768);
var kv=Cache(l,g,d/h,32768);
Console.WriteLine($"{name} Q={h} KV={g} MHA={mha/(double)(1L<<30):F2} Gio livré={kv/(double)(1L<<30):F2} Gio rapport={mha/(double)kv:F1}");
}
var cache=torch.zeros(new long[]{1,8,4096,128},dtype:ScalarType.Float16);
long measured=2*cache.numel()*2;
 Console.WriteLine($"Cache FP16 K/V : mesuré {measured}, formule {Cache(1,8,128,4096)} octets");
if(measured!=Cache(1,8,128,4096))throw new Exception("cache");


LLaMA-2 7B Q=32 KV=32 MHA=16.00 Gio livré=16.00 Gio rapport=1.0


LLaMA-2 70B Q=64 KV=8 MHA=80.00 Gio livré=10.00 Gio rapport=8.0


Mistral 7B Q=32 KV=8 MHA=16.00 Gio livré=4.00 Gio rapport=4.0


notre jouet Q=4 KV=4 MHA=0.02 Gio livré=0.02 Gio rapport=1.0


Cache FP16 K/V : mesuré 16777216, formule 16777216 octets


### 2.1 Lecture

Le cache comporte deux tenseurs K et V. Le tableau utilise FP16 et le même contexte ; le contrôle final compare formule et taille tensorielle.

## 3. GQA et MQA : partager les tetes KV

MQA et GQA ne sont pas deux mécanismes, c'est **un seul** muni d'un paramètre. On ecrit une classe
qui prend `n_kv_heads` et couvre les trois regimes :

| `n_kv_heads` | Nom | Ce que fait la tete Q `h` |
|---|---|---|
| `n_heads` | MHA | lit sa propre paire K/V |
| `1` | MQA | lit l'unique paire K/V, comme toutes les autres |
| `1 < g < n_heads` | GQA | lit la paire K/V du groupe `h // (n_heads / g)` |

L'etagement se fait par `repeat_interleave`, qui **duplique** la vue sans copier les poids : la
tete Q `h` recoit la tete KV d'indice `h // rep` avec `rep = n_heads / n_kv_heads`. MQA est donc
exactement GQA avec `n_kv_heads = 1`. Ecrire les deux separement serait ecrire deux fois le même code
et se donner deux fois l'occasion de les faire diverger.

Le partage reduit aussi le **nombre de paramètres** des projections K et V, d'un facteur
`n_heads / n_kv_heads` : `W_k` et `W_v` passent de `(n_heads * dh, d_model)` a
`(n_kv_heads * dh, d_model)`. Ce n'est pas le but recherche — c'est un effet de bord a connaitre,
parce qu'il rend toute comparaison de qualite entre variantes **boiteuse a paramètres inegaux**, et
qu'il faudra donc l'annoncer plutot que de la taire.

In [5]:
var kv0=torch.randn(new long[]{1,2,5,4});
var expanded=kv0.repeat_interleave(4,1);
Console.WriteLine($"(d) forme : {string.Join(",",expanded.shape)}");
for(int h=0;h<8;h++){var d=(expanded.select(1,h)-kv0.select(1,h/4)).abs().max().item<float>();
if(d!=0)throw new Exception("groupement");
}Console.WriteLine("Chaque tête Q lit la tête KV attendue.");

foreach(int g in new[]{32,8,1}){using var scope=torch.NewDisposeScope();
using var m=new Variant(4096,32,g);
long expected=2L*4096*g*128;
var actual=m.KvParams();
 if(actual!=expected)throw new Exception("compte K/V");
 Console.WriteLine($"groupes {g} : K/V {actual:N0} paramètres ; total module {m.parameters().Sum(p=>p.numel()):N0}");
}

(d) forme : 1,8,5,4


Chaque tête Q lit la tête KV attendue.


groupes 32 : K/V 33,554,432 paramètres ; total module 67,108,864


groupes 8 : K/V 8,388,608 paramètres ; total module 41,943,040


groupes 1 : K/V 1,048,576 paramètres ; total module 34,603,008


### 3.1 Lecture

Le partage répète chaque tête KV pour un groupe de têtes Q. Les paramètres K/V diminuent avec le nombre de groupes ; les autres projections subsistent.

## 4. SWA : la bande locale

L'attention a fenêtre glissante restreint chaque requête aux `W` dernières clés. Le masque est
l'intersection de deux conditions : causale (`j <= i`) et locale (`i - j < W`). Le comptage de la
verification `(c)` dit déjà l'essentiel : `T*W - W(W-1)/2` paires au lieu de `T(T+1)/2`.

Il y a cependant **deux facons de coder** cette restriction, et la différence entre les deux est le
sujet d'une bonne partie de la section 5 :

- **par masque** : on calcule les `T x T` scores comme d'habitude, puis on remplace par `-inf` tout ce
  qui sort de la bande. C'est direct, lisible, et le cout reste `O(T^2)` : on a calcule les scores
  hors bande pour les jeter. Le gain memoire du cache est toujours la, le gain de calcul est nul.
- **par bande** : on ne materialise que les `W` colonnes utiles en decalant les clés, ce qui donne un
  tenseur `(B, H, T, W)`. Le cout devient reellement `O(T*W)`, mais plus aucune opération n'est un
  produit matriciel dense — et c'est précisément la le piege que la section 5 va mesurer.

Les deux doivent donner le même résultat. C'est la verification `(e)` : elle autorise a utiliser la
bande pour les mesures de vitesse sans avoir a croire sa correction sur parole.

In [6]:
foreach(var (T,W) in new[]{(64,16),(64,33),(128,5),(37,37)}){using var scope=torch.NewDisposeScope();
var a=torch.randn(new long[]{2,4,T,8});
var b=torch.randn(new long[]{2,4,T,8});
var c=torch.randn(new long[]{2,4,T,8});
var x=Variant.Mask(a,b,c,W);
var y=Variant.Band(a,b,c,W);
var err=(x-y).abs().max().item<float>();
Console.WriteLine($"T={T}, W={W}, masque/bande {err:E3}");
if(err>1e-5)throw new Exception("bande incorrecte");
}
torch.manual_seed(0);
var m1=new Variant(64,4,4,16,false);
var m2=new Variant(64,4,4,16,true);
m2.load_state_dict(m1.state_dict());

var x1=torch.randn(new long[]{2,32,64},requires_grad:true);
var x2=x1.detach().clone().requires_grad_(true);
var y1=m1.forward(x1);
var y2=m2.forward(x2);
var forward=(y1-y2).abs().max().item<float>();
y1.sum().backward();
y2.sum().backward();
var grad=(x1.grad-x2.grad).abs().max().item<float>();
Console.WriteLine($"même module : avant {forward:E3}, gradient {grad:E3}");
if(forward>1e-5 || grad>1e-5)throw new Exception("écart module/gradient");


T=64, W=16, masque/bande 3.576E-007


T=64, W=33, masque/bande 4.768E-007


T=128, W=5, masque/bande 3.576E-007


T=37, W=37, masque/bande 4.768E-007


même module : avant 1.192E-007, gradient 3.576E-007


### 4.1 Lecture

La bande et le masque sont confrontés sur plusieurs tailles, puis au niveau du module avec les mêmes poids et gradients.

## Exercice 1 — le groupement des tetes Q

**Objectif.** Ecrire le groupement des tetes KV sans `repeat_interleave`, avec un `view` (ou un
`reshape` suivi d'un `expand`) : c'est la forme qu'utilisent les implementations industrielles, parce
qu'elle evite le tenseur intermediaire.

**Indice.** Les `n_heads` tetes Q se lisent comme `G` groupes de `rep` tetes. Une tete Q d'indice `h`
appartient au groupe `h // rep`. On peut donc passer de `(B, G, T, dh)` a `(B, G, 1, T, dh)` puis
etaler le groupe sur `rep` — sans jamais copier les données.

**Étape 1.** Redimensionner le tenseur KV de `(B, G, T, dh)` en `(B, G, 1, T, dh)`.
**Étape 2.** L'etaler sur l'axe `rep` avec `expand`, puis remettre a plat en `(B, G * rep, T, dh)`.
**Étape 3.** Verifier contre `build_kv_heads` : l'ecart doit etre exactement nul (même adresse memoire,
pas de recopie) et la forme doit etre `(B, n_heads, T, dh)`.

**Pourquoi ca compte.** Le `expand` ne materialise pas de copie ; `repeat_interleave` non plus, mais
il construit un nouveau tenseur de même forme finale. Sur un cache KV de plusieurs gigaoctets, la
différence entre « voir » et « copier » est la différence entre une optimisation et une regression.

In [7]:
Tensor BuildKvHeadsParVue(Tensor kv,int heads,int groups){ // TODO : view, expand puis reshape ; vérifier le stockage et les valeurs.
 return null;

}
var kvEssai=torch.randn(new long[]{2,2,6,4});
 var attendu=kvEssai.repeat_interleave(4,1);
var obtenu=BuildKvHeadsParVue(kvEssai,8,2);

if(obtenu is null) Console.WriteLine("Exercice 1 à compléter : résultat null");

else {Console.WriteLine($"forme {string.Join(",",obtenu.shape)}, écart {(obtenu-attendu).abs().max().item<float>()}");
var before=obtenu.clone();
 kvEssai.add_(1);
 var share=(obtenu-before).abs().max().item<float>()>0;
 kvEssai.sub_(1);
 Console.WriteLine($"stockage partagé (test de mutation) : {share}");
}

Exercice 1 à compléter : résultat null


## 5. Vitesse : ce que le partage accelere, et ce qu'il n'accelere pas

On mesure d'abord le cout reel d'un pas forward + backward d'une couche d'attention, pour les quatre
variantes, a deux longueurs de sequence. Puis on isole la question de la fenêtre en comparant les deux
implementations de la section 4 sur un balayage de `T`.

**Trois avertissements de méthode, avant de lire quoi que ce soit.**

Le premier est que ce banc tourne sur **un seul thread CPU**. Ce n'est pas un detail de confort :
c'est ce qui rend les mesures reproductibles, et c'est aussi ce qui rend les conclusions non
transposables telles quelles a un GPU. Sur GPU, l'argument du partage des tetes KV est un argument de
**bande passante memoire** — le cache est lu a chaque jeton genere, et un cache huit fois plus petit
est un cache lu huit fois plus vite. Cette contrainte n'existe pas de la même facon dans un CPU
mono-thread qui execute du code TorchSharp.

Le second est qu'on mesure ici un **pas d'entraînement** : sequence complete, toutes les requêtes en
parallele. C'est le regime ou le partage des tetes KV n'a presque rien a apporter, et le mesurer
honnetement est plus instructif que de le supposer. Le regime ou il apporte tout est celui de la
**generation**, ou l'on traite un jeton a la fois et ou le cache est relu integralement — regime que
ce banc ne simule pas, et qu'on ne pretend donc pas mesurer.

Le troisieme est empirique, et c'est celui qui a change ce notebook. Un banc qui chronometre les
variantes **a la suite** ne mesure pas seulement la variante : il mesure aussi sa **position** dans
la boucle, la première payant l'allocation des tampons. Les mesures sont donc **entrelacees**, et le
tableau imprime un **contrôle** entre deux variantes dont la computation est identique — sans quoi
aucun des ecarts lus plus bas ne voudrait dire quoi que ce soit.


In [8]:
Dictionary<string,double> Bench(List<(string,Variant)> modules,int T,int batch=2,int warm=3,int rounds=9){
 using var x=torch.randn(new long[]{batch,T,128},requires_grad:true);
foreach(var (_,m) in modules)for(int i=0;i<warm;i++){using var scope=torch.NewDisposeScope();
m.forward(x).sum().backward();
}
 var samples=modules.ToDictionary(p=>p.Item1,p=>new List<double>());

 for(int i=0;i<rounds;i++)foreach(var (name,m) in modules){using var scope=torch.NewDisposeScope();
m.zero_grad();
var sw=Stopwatch.StartNew();
m.forward(x).sum().backward();
sw.Stop();
samples[name].Add(sw.Elapsed.TotalMilliseconds);
}
 return samples.ToDictionary(p=>p.Key,p=>p.Value.OrderBy(x=>x).ElementAt(rounds/2));

}
Variant Make(int groups,int? window=null,bool band=false){torch.manual_seed(0);
return new Variant(128,4,groups,window,band);
}
foreach(int T in new[]{128,512}){var mods=new List<(string,Variant)>{("MHA",Make(4)),("GQA",Make(2)),("MQA",Make(1)),("SWA",Make(4,128))};
var t=Bench(mods,T);
Console.WriteLine($"T={T}: "+string.Join(" | ",t.Select(x=>$"{x.Key} {x.Value:F2} ms")));
Console.WriteLine($"SWA/MHA {t["SWA"]/t["MHA"]:F2}, GQA/MHA {t["GQA"]/t["MHA"]:F2}, MQA/MHA {t["MQA"]/t["MHA"]:F2}");
foreach(var (_,m) in mods){Console.WriteLine($"paramètres {m.parameters().Sum(p=>p.numel())}");
m.Dispose();
}}

T=128: MHA 4.02 ms | GQA 3.79 ms | MQA 3.53 ms | SWA 4.11 ms


SWA/MHA 1.02, GQA/MHA 0.94, MQA/MHA 0.88


paramètres 65536


paramètres 49152


paramètres 40960


paramètres 65536


... (2 outputs left out)

paramètres 65536


paramètres 49152


paramètres 40960


paramètres 65536


La mesure suivante prolonge le banc précédent avec les mêmes conditions.

In [9]:
foreach(int T in new[]{128,512,1024}){var mods=new List<(string,Variant)>{("masque",Make(4,128)),("bande",Make(4,128,true))};
var t=Bench(mods,T,batch:1);
var p=T*(T+1)/2;
var band=Enumerable.Range(1,T).Sum(i=>Math.Min(i,128));
Console.WriteLine($"T={T} masque {t["masque"]:F2} ms, bande {t["bande"]:F2} ms, rapport {t["masque"]/t["bande"]:F2}, paires {p}/{band}");
foreach(var (_,m) in mods)m.Dispose();
}

T=128 masque 1.82 ms, bande 14.58 ms, rapport 0.13, paires 8256/8256


T=512 masque 21.29 ms, bande 162.87 ms, rapport 0.13, paires 131328/57408


T=1024 masque 142.42 ms, bande 306.84 ms, rapport 0.46, paires 524800/122944


### 5.1 Lecture

Les médianes sont mesurées en tours entrelacés, trois chauffes et neuf répétitions. Les rapports sont propres à ce replay CPU et ne prouvent pas un gain universel. Le contrôle W ≥ T compare la même computation.

Les comptes de paramètres après chaque tableau suivent le même ordre que les mesures : MHA, GQA, MQA, puis SWA. Le contrôle W ≥ T compare deux calculs identiques mais leurs temps peuvent différer sous contention. La bande explicite évite des paires, ce qui ne suffit pas à garantir une latence inférieure : allocation, accès mémoire et noyaux utilisés comptent aussi.

## Exercice 2 — le masque en bande, ecrit a la main

**Objectif.** Reconstruire le masque de la bande `(1, 1, T, W)` sans utiliser la condition
`decalage >= W - 1 - position`, et sans boucle sur `T`.

**Indice.** La position absolue lue par la requête `i` a l'offset `d` vaut `i + d - (W - 1)`. La
condition de validite est qu'elle soit `>= 0`. C'est la même chose que de dire que le masque est un
triangle degenere : pour la requête `i`, les premiers `max(0, W - 1 - i)` offsets sont invalides, les
autres sont valides. Une comparaison de deux tenseurs crees par `arange`, diffuses l'un contre
l'autre, suffit.

**Étape 1.** Construire `decalages = torch.arange(W)` de forme `(W,)` et `positions = torch.arange(T)`
de forme `(T,)`.
**Étape 2.** Former `decalages[None, :] >= (W - 1 - positions)[:, None]`, de forme `(T, W)`.
**Étape 3.** Verifier contre `causal_window_mask(T, W)` : a l'offset `d`, la cle lue est la position
`i + d - (W - 1)`. Le nombre de `True` par ligne doit valoir `min(i + 1, W)`, et la somme totale
`T*W - W(W-1)/2`.

**Pourquoi ca compte.** Cette matrice est le seul endroit ou la bande peut etre fausse sans que rien
ne leve d'erreur : un decalage d'un cran produit un modèle qui apprend sur des clés decalees, avec des
metriques plausibles et un résultat faux.

In [10]:
Tensor MasqueBande(int T,int W){ // TODO : deux arange et une comparaison diffusée
 return null;

}
var mb=MasqueBande(64,16);
if(mb is null)Console.WriteLine("Exercice 2 à compléter : résultat null");
else{var expectedCounts=torch.tensor(Enumerable.Range(0,64).Select(i=>(long)Math.Min(i+1,16)).ToArray());
Console.WriteLine($"forme {string.Join(",",mb.shape)}, écart comptes {(mb.sum(1)-expectedCounts).abs().max().item<long>()}, total {mb.sum().item<long>()}, formule {64*16-16*15/2}");
}

Exercice 2 à compléter : résultat null



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy



## 6. Qualite : une tâche ou la fenêtre se paie cash

Le cout de la fenêtre ne se lit pas dans un chiffre de perplexite sur un corpus quelconque : il se lit
sur une tâche ou l'information utile est **au-dela de la fenêtre**. On construit donc un corpus dont
la dépendance longue est plantee et verifiable.

**La tâche.** Chaque sequence commence par un marqueur tire parmi 8, suivi de jetons de remplissage,
et se termine par un jeton de requête. La cible est le **marqueur initial** : le modèle doit le
retrouver a la dernière position, après `T - 1 = 63` jetons d'ecart. Huit marqueurs equiprobables
fixent le hasard a `1/8 = 0.125` en exactitude, et a une perplexite de `8` a la position de requête.

**Pourquoi cette tâche et pas un corpus naturel.** Elle rend la grandeur mesuree **interpretable par
construction** : sous la fenêtre `W`, une requête situee a plus de `W` jetons du marqueur ne peut
litteralement pas le voir, donc la borne superieure atteignable est le hasard. Un corpus naturel
n'aurait pas cette propriete, et une degradation de perplexite y serait attribuable a dix causes
melangees. On perd en realisme ce qu'on gagne en attribution.

**Trois contrôles sont poses dans le dispositif**, parce qu'un résultat negatif sans contrôle ne
prouve rien :

1. une variante a fenêtre `W = T = 64` : la fenêtre couvre alors toute la sequence (verification
   `(b)`), donc le chemin de code SWA est exerce sans restriction. S'il apprend, un echec des
   fenêtres courtes est attribuable a la restriction, pas a un bug ;
2. MHA, GQA et MQA : trois facons de lire toute la sequence, qui doivent toutes apprendre ;
3. la memetisation par profondeur : une fenêtre courte dans une pile de couches donne un champ
   receptif `L * (W - 1) + 1`. Avec `W = 16`, il faut `L >= 5` pour couvrir 64 positions. Le test
   mesure si cette théorie se realise.

On rapporte l'exactitude **et** la perplexite a la position de requête — pas la perplexite globale,
qui serait dominee par les jetons de remplissage, uniformement aléatoires et donc non apprenables.

In [11]:
public class Bloc:Module<Tensor,Tensor>{
 readonly Module<Tensor,Tensor> ln1,ln2,up,down,act;
 readonly Variant attn;

 public Bloc(int d,int h,int g,int? w):base("Bloc"){ln1=torch.nn.LayerNorm(new long[]{d});
ln2=torch.nn.LayerNorm(new long[]{d});
attn=new Variant(d,h,g,w);
up=torch.nn.Linear(d,4*d);
down=torch.nn.Linear(4*d,d);
act=torch.nn.GELU();
RegisterComponents();
}
 public override Tensor forward(Tensor x){var y=x+attn.forward(ln1.forward(x));
return y+down.forward(act.forward(up.forward(ln2.forward(y))));
}
}
public class Petit:Module<Tensor,Tensor>{
 readonly Module<Tensor,Tensor> emb,ln,head;
readonly Bloc b0,b1,b2,b3,b4,b5;
readonly int layers;

 public Petit(int vocab,int d,int h,int g,int? w,int L):base("Petit"){layers=L;
emb=torch.nn.Embedding(vocab,d);
ln=torch.nn.LayerNorm(new long[]{d});
head=torch.nn.Linear(d,vocab,hasBias:false);
b0=new Bloc(d,h,g,w);
if(L>1)b1=new Bloc(d,h,g,w);
if(L>2)b2=new Bloc(d,h,g,w);
if(L>3)b3=new Bloc(d,h,g,w);
if(L>4)b4=new Bloc(d,h,g,w);
if(L>5)b5=new Bloc(d,h,g,w);
RegisterComponents();
}
 public override Tensor forward(Tensor x){var y=emb.forward(x);
y=b0.forward(y);
if(layers>1)y=b1.forward(y);
if(layers>2)y=b2.forward(y);
if(layers>3)y=b3.forward(y);
if(layers>4)y=b4.forward(y);
if(layers>5)y=b5.forward(y);
return head.forward(ln.forward(y));
}
}

const int VOCAB=19;
const int T=64;

Tensor Lot(int n,torch.Generator gen){var marker=torch.randint(0,8,new long[]{n,1},generator:gen);
var fill=torch.randint(8,18,new long[]{n,T-2},generator:gen);
var query=torch.full(new long[]{n,1},18,dtype:ScalarType.Int64);
return torch.cat(new[]{marker,fill,query},1);
}
(double,double,double,long) Entrainer(int? window,int layers,int seed,int steps=200,int groups=4){torch.manual_seed(seed);
using var model=new Petit(VOCAB,64,4,groups,window,layers);
using var opt=torch.optim.Adam(model.parameters(),lr:0.003);
var gen=new torch.Generator((ulong)(1234+seed),torch.CPU);
var sw=Stopwatch.StartNew();
for(int i=0;i<steps;i++){using var scope=torch.NewDisposeScope();
var x=Lot(32,gen);
var logits=model.forward(x).select(1,T-1);
var loss=torch.nn.functional.cross_entropy(logits,x.select(1,0));
opt.zero_grad();
loss.backward();
opt.step();
}sw.Stop();
using var evalScope=torch.NewDisposeScope();
using var ng=torch.no_grad();
var test=Lot(512,new torch.Generator(99,torch.CPU));
var output=model.forward(test).select(1,T-1);
var lossTest=torch.nn.functional.cross_entropy(output,test.select(1,0)).item<float>();
var acc=output.argmax(-1).eq(test.select(1,0)).to_type(ScalarType.Float32).mean().item<float>();
return (acc,Math.Exp(lossTest),sw.Elapsed.TotalSeconds,model.parameters().Sum(p=>p.numel()));
}
Console.WriteLine("Rappel à distance 63, hasard 1/8, perplexité 8 ; 200 pas, graines 0 et 1.");

foreach(var (name,g,w) in new (string,int,int?)[]{("MHA",4,null),("GQA",2,null),("MQA",1,null),("SWA W8",4,8),("SWA W16",4,16),("SWA W32",4,32),("SWA W64",4,64)}){var a=Entrainer(w,2,0,groups:g);
var b=Entrainer(w,2,1,groups:g);
Console.WriteLine($"{name} accuracy {(a.Item1+b.Item1)/2:F3} ± {Math.Abs(a.Item1-b.Item1)/2:F3}, ppl {(a.Item2+b.Item2)/2:F2}, secondes {b.Item3:F1}, paramètres {b.Item4}");
}

Rappel à distance 63, hasard 1/8, perplexité 8 ; 200 pas, graines 0 et 1.


MHA accuracy 1.000 ± 0.000, ppl 1.00, secondes 9.3, paramètres 102016


GQA accuracy 1.000 ± 0.000, ppl 1.00, secondes 9.3, paramètres 93824


MQA accuracy 1.000 ± 0.000, ppl 1.00, secondes 8.9, paramètres 89728


SWA W8 accuracy 0.126 ± 0.001, ppl 8.09, secondes 7.2, paramètres 102016


SWA W16 accuracy 0.126 ± 0.001, ppl 8.08, secondes 8.8, paramètres 102016


SWA W32 accuracy 0.126 ± 0.001, ppl 8.08, secondes 9.7, paramètres 102016


SWA W64 accuracy 1.000 ± 0.000, ppl 1.00, secondes 8.9, paramètres 102016


La mesure suivante prolonge le banc précédent avec les mêmes conditions.

In [12]:
Console.WriteLine("Champ réceptif : L*(W-1)+1 ; distance 63 ; W=16.");
foreach(int L in new[]{1,2,4,6}){var a=Entrainer(16,L,0,steps:400);
Console.WriteLine($"L={L}, champ {L*15+1}, accuracy {a.Item1:F3}, ppl {a.Item2:F2}, secondes {a.Item3:F1}");
}

Champ réceptif : L*(W-1)+1 ; distance 63 ; W=16.


L=1, champ 16, accuracy 0.141, ppl 8.11, secondes 8.8


L=2, champ 31, accuracy 0.141, ppl 8.12, secondes 16.3


L=4, champ 61, accuracy 0.141, ppl 8.12, secondes 29.4


L=6, champ 91, accuracy 0.141, ppl 8.13, secondes 45.8


### 6.1 Lecture

Le rappel exige de transmettre le marqueur initial jusqu’à la requête finale. Comparez les fenêtres, la causalité complète et les champs réceptifs mesurés ci-dessus ; deux graines pour les variantes et une graine pour la profondeur ne suffisent pas à établir une loi générale.

Sur ce replay, MHA, GQA, MQA et la fenêtre couvrant toute la séquence retrouvent le marqueur. Les fenêtres courtes restent proches du hasard `1/8`. Augmenter la profondeur jusqu’à six couches rend le champ théorique supérieur à la distance, mais ne suffit pas à apprendre la tâche dans ce budget : la possibilité de transmettre une information ne garantit pas son apprentissage. La comparaison de profondeur repose sur une seule graine.

## Exercice 3 — le point de croisement, mesure

**Objectif.** Trouver par la mesure la longueur `T` a partir de laquelle la bande explicite devient
plus rapide que le masque, pour une fenêtre `W = 64`.

**Indice.** Le point de croisement depend de `T/W` et de l'efficacite relative du `einsum` a cinq
dimensions contre le produit matriciel dense. Chercher la longueur ou les deux courbes se croisent
revient a chercher `T` tel que `T^2 ~ c * T * W`, avec `c` le surcout par élément de la bande.

**Étape 1.** Choisir au moins cinq longueurs croissantes, dont une sous `512` et une au-dessus de
`1024`.
**Étape 2.** Mesurer forward + backward des deux implementations pour chacune (helper `bench_module`).
**Étape 3.** Rapporter le rapport `masque / bande` et conclure sur l'ordre de grandeur du croisement.

**Pourquoi ca compte.** C'est la mesure qui separe « l'algorithme est asymptotiquement meilleur » de
« ce code-ci est plus rapide ». Un `O(T*W)` ecrit avec des opérations inefficaces peut rester plus
lent que le `O(T^2)` qu'il remplace sur toute la plage utile, et seul un chronometrage permet de le
savoir.

Pour reprendre le jeu proposé, utilisez `128, 256, 512, 1024, 1536` lorsque `lengths` est nul. La fonction reste à compléter.

In [13]:
Dictionary<int,(double,double)> Croisement(int W=64,int[] lengths=null){ // TODO : mesurer les deux modules ensemble avec Bench, pour chaque longueur
 return null;

}
var measures=Croisement();
if(measures is null)Console.WriteLine("Exercice 3 à compléter : résultat null");
else foreach(var (length,t) in measures)Console.WriteLine($"{length} masque {t.Item1:F2}, bande {t.Item2:F2}, gain {t.Item1/t.Item2:F2}");


Exercice 3 à compléter : résultat null


## 7. Conclusion

Le cache dépend du nombre de têtes KV, tandis que la fenêtre limite les positions visibles. Les vérifications de calcul et gradients précèdent les mesures de temps et le test de rappel. Les résultats numériques et leurs limites sont ceux du replay ci-dessus.

[RoPE C#](TV-00a-RoPE-from-scratch-CSharp.ipynb) | [Suite : encodeurs Python](TV-00d-Encodeurs-from-scratch.ipynb)